# O nó de opressões no ENEM 2025
## Raça, gênero e classe como sistema articulado de dominação-exploração

**Referencial teórico:** Feminismo marxista | Heleieth Saffioti  
**Dados:** Microdados ENEM 2025 — INEP/MEC

### ⚠️ Nota metodológica importante

A partir de 2020, o INEP dividiu os microdados em dois arquivos (PARTICIPANTES e RESULTADOS) por exigência da LGPD, **sem chave de junção individual**. Isso significa que **não é possível vincular dados demográficos (raça, gênero) a notas individuais**. A análise a seguir é **ecológica** (nível UF): correlações agregadas não implicam relações individuais.

---

### Por que "nó de opressões" e não "interseccionalidade"?

A metáfora da intersecção pressupõe eixos que se cruzam. O **nó de opressões** (Saffioti, 2004) recusa essa separabilidade: raça, gênero e classe se constituem mutuamente dentro de um sistema único de dominação-exploração patriarcal-racista-capitalista.

- A mulher negra não é "mulher + negra": ela ocupa uma **posição estrutural** específica.
- Não somamos desvantagens; identificamos configurações do nó.
- A unidade de análise são os **grupos estruturais**, não atributos cruzados.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import sys
sys.path.append('../src')

from constants import GRUPOS_PRINCIPAIS, ORDEM_GRUPOS, CORES, ANO
from visualizacoes import (
    grafico_composicao_demografica,
    grafico_notas_por_tipo_escola,
    grafico_ausencia_por_tipo,
    heatmap_composicao_uf,
    grafico_gap_escola,
)

pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_columns', 40)

print(f'Ambiente configurado. ENEM {ANO}.')

## 1. Carregamento dos dados processados

Os dados já foram limpos e processados por `src/ingestao.py` e `src/grupos.py`.  
Se ainda não rodou: `python src/ingestao.py && python src/grupos.py && python src/inferencial.py`

In [ ]:
# Dados de participantes (demografia)
df_part = pd.read_parquet('../data/processed/participantes_limpo.parquet')
# Dados de resultados (notas e presença)
df_res = pd.read_parquet('../data/processed/resultados_limpo.parquet')
# Métricas agregadas
df_demo = pd.read_parquet('../data/processed/metricas_demograficas.parquet')
df_desemp = pd.read_parquet('../data/processed/metricas_desempenho.parquet')
df_gaps = pd.read_parquet('../data/processed/gaps_regionais.parquet')
df_coefs = pd.read_parquet('../data/processed/regressao_ecologica.parquet')

print(f'Participantes: {len(df_part):,}')
print(f'Resultados: {len(df_res):,}')
print(f'UFs com métricas demográficas: {df_demo["uf"].nunique()}')
print(f'UFs com métricas de desempenho: {df_desemp["uf"].nunique()}')

## 2. Quem se inscreve? Composição demográfica por UF

A **composição demográfica** dos inscritos revela a estrutura social desigual que antecede o exame: onde a população negra é maioria, os recursos educacionais tendem a ser mais escassos.

In [ ]:
# Distribuição dos grupos estruturais
grupos_n = df_part['grupo_no'].value_counts().reset_index()
grupos_n.columns = ['grupo', 'n']
grupos_n['%'] = (grupos_n['n'] / len(df_part) * 100).round(1)
print('Distribuição por grupo estrutural (nó raça × gênero):')
print(grupos_n.to_string(index=False))

In [ ]:
# Visualização: composição demográfica por UF
fig = grafico_composicao_demografica(df_demo)
fig.show()

In [ ]:
# Heatmap: raça × gênero por UF
fig = heatmap_composicao_uf(df_demo)
fig.show()

### Interpretação

> A composição demográfica dos inscritos não é aleatória: ela reflete a estrutura social do país. Onde a população negra é majoritária, os investimentos educacionais tendem a ser menores. O nó de opressões se manifesta já na possibilidade de se inscrever — e nas condições materiais que determinam quem chega ao exame.

## 3. Quem consegue chegar? Ausência por tipo de escola

A ausência ao exame é um **indicador estrutural**: quem não chega à prova já estava excluído antes dela.

In [ ]:
# Taxa de presença geral
total = len(df_res)
presentes = df_res['presente_ambos_dias'].sum()
print(f'Total de inscritos: {total:,}')
print(f'Presentes em ambos os dias: {presentes:,} ({presentes/total*100:.1f}%)')
print(f'Ausentes em pelo menos um dia: {total - presentes:,} ({(total-presentes)/total*100:.1f}%)')
print()
print('Ausência por tipo de escola:')
print(df_res.groupby('escola_tipo')['ausente_algum_dia'].mean().round(3) * 100)

In [ ]:
# Visualização: ausência por tipo de escola
from visualizacoes import grafico_ausencia_por_tipo
presenca_tipo = df_res.groupby(['uf', 'escola_tipo']).agg(
    n_total=('presente_ambos_dias', 'size'),
    n_presentes=('presente_ambos_dias', 'sum'),
).reset_index()
presenca_tipo['pct_ausente'] = (1 - presenca_tipo['n_presentes']/presenca_tipo['n_total']) * 100
fig = grafico_ausencia_por_tipo(presenca_tipo[presenca_tipo['escola_tipo'].isin(['Pública', 'Privada'])])
fig.show()

### Interpretação

> Escolas públicas concentram a maior taxa de ausência — resultado da interação de opressões que vai além da renda individual. A ausência diferenciada por tipo de escola é dado estrutural: antes de qualquer nota, a estrutura já age.

## 4. Desempenho por tipo de escola e UF

O gap entre escola pública e privada não é apenas de recursos: é um gap estrutural que se reproduz porque a escola pública atende majoritariamente a população oprimida pelo nó.

In [ ]:
# Gap entre escola pública e privada
print(f'Gap médio entre escola pública e privada: {df_gaps["gap_publico_privado"].mean():.1f} pontos')
print(f'UF com menor gap: {df_gaps.loc[df_gaps["gap_publico_privado"].idxmin(), "uf"]} ({df_gaps["gap_publico_privado"].min():.1f} pontos)')
print(f'UF com maior gap: {df_gaps.loc[df_gaps["gap_publico_privado"].idxmax(), "uf"]} ({df_gaps["gap_publico_privado"].max():.1f} pontos)')
print()
print('Top 5 UFs com maior gap:')
print(df_gaps.nlargest(5, 'gap_publico_privado')[['uf', 'gap_publico_privado', 'nota_media_uf']].to_string(index=False))

In [ ]:
# Visualização: gap por UF
fig = grafico_gap_escola(df_gaps.dropna(subset=['gap_publico_privado']))
fig.show()

## 5. Análise ecológica: composição demográfica e desempenho

> ⚠️ **Efeito ecológico**: Estas correlações são no nível de UF, não individual. Ver Saffioti (2004) para a articulação teórica.

In [ ]:
# Coeficientes da regressão ecológica
if not df_coefs.empty:
    print('Regressão ecológica: nota_media_uf ~ pct_mulher_negra + renda_media + pct_presente_uf')
    print(f'R² = {df_coefs["r2"].iloc[0]:.3f}')
    print()
    print(df_coefs[['variavel', 'coef', 'std_err', 'p_valor']].to_string(index=False))

In [ ]:
# Visualização: coeficientes ecológicos
if not df_coefs.empty:
    from visualizacoes import grafico_coeficientes_ecologicos
    fig = grafico_coeficientes_ecologicos(df_coefs)
    fig.show()

### Interpretação

> A regressão ecológica estima a associação entre composição demográfica regional e desempenho médio regional. UFs com maior proporção de mulheres negras tendem a ter notas médias menores — não porque mulheres negras individualmente têm notas menores, mas porque a concentração de população oprimida indica regiões com menores investimentos educacionais.

> O R² de 0.86 indica que a composição demográfica, a renda e a presença explicam 86% da variação regional de notas — evidência forte do nó operando nos dados.

## 6. Conclusões analíticas

**O que os dados do ENEM 2025 nos dizem sobre o nó (nível ecológico)?**

1. **Composição demográfica é preditora de desempenho**: UFs com maior proporção de população oprimida têm notas médias significativamente menores.

2. **O gap público vs privado é estrutural**: em todos os 27 UFs, escolas privadas têm notas médias superiores — com gaps de até 137 pontos.

3. **A ausência é o primeiro nó**: antes de qualquer nota, a taxa de ausência diferenciada por tipo de escola revela quem o sistema exclui antes da porta.

4. **Classe amplifica, mas não esgota**: a renda média regional explica parte da variação, mas a composição demográfica tem efeito autônomo — articulado, não redutível.

**Limitações metodológicas**

- **Análise ecológica**: não podemos vincular raça/gênero a notas individuais (LGPD). Correlações agregadas não implicam relações individuais.
- Raça/cor por autodeclaração: captura identidade, não determinação racial objetiva.
- Gênero binário no questionário INEP: apaga pessoas trans e não-binárias.
- Tipo de escola com ~64% de "Não respondeu" (TP_DEPENDENCIA_ADM_ESC ausente para muitos inscritos).